# vLLM_Real_Time — Test Runner

Waits for the vLLM server to be ready on the driver proxy, then sends TC1–TC4 as image inference requests and measures per-document latency.

**This notebook does NOT run vLLM** — it sends HTTP requests to the already-running vLLM_Real_Time continuous job.

### Cluster Requirements

| Setting | Value |
|---------|-------|
| Instance | Serverless |
| Libraries | None — installed via `%pip` below |

### Prerequisites
- `setup/01_prepare_test_cases` — TC PDFs must exist in Volume
- `vllm_real_time/notebook` continuous job must be **RUNNING** (not paused)
- Set the `cluster_id` widget to the RT job's cluster ID

### Widgets
- **`cluster_id`** (required): Cluster ID where vLLM_Real_Time is running (find in Job → Run → Cluster tab)

### Install Dependencies

In [ ]:
%pip install pymupdf requests pyyaml

### Configuration

Reads `config.yaml` for Volume paths, vLLM port, and model name. The driver proxy URL is constructed from the `cluster_id` widget.

**Key timeouts:**
- `READY_TIMEOUT = 1200s` — max wait for vLLM to start responding (includes model loading after cluster start)
- `TEST_TIMEOUT = 300s` — per-request timeout (first image inference triggers CUDA kernel JIT compilation, which can take >120s)

In [ ]:
import yaml, os, base64, time, requests, fitz
from datetime import datetime, timezone

# Resolve project root
if "__file__" in dir():
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + os.path.dirname(os.path.dirname(_nb))

cfg           = yaml.safe_load(open(f"{_root}/config.yaml"))
CATALOG       = cfg["catalog"]
SCHEMA        = cfg["schema"]
TC_PATH       = f"/Volumes/{CATALOG}/{SCHEMA}/{cfg['volume']}/{cfg['test_cases_subpath']}"
PERF_TABLE    = f"{CATALOG}.{SCHEMA}.{cfg['perf_table']}"
RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.{cfg['rt_results_table']}"
PORT          = cfg["vllm_port"]
MODEL_NAME    = cfg["vllm_model_name"]

# Workspace URL and auth token for driver proxy requests
workspace  = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
token      = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Build the driver proxy URL from the cluster_id widget
dbutils.widgets.text("cluster_id", "")
cluster_id = dbutils.widgets.get("cluster_id")
BASE_URL   = f"{workspace}/driver-proxy-api/o/0/{cluster_id}/{PORT}/v1"
HEADERS    = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Timeouts
READY_TIMEOUT = 1200  # Max wait for vLLM health check to return 200
TEST_TIMEOUT  = 300   # Per-request timeout (first inference triggers CUDA JIT)

print(f"Proxy URL     : {BASE_URL}")
print(f"Model name    : {MODEL_NAME}")
print(f"Results table : {RESULTS_TABLE}")
print(f"TC path       : {TC_PATH}")

### Ensure Tables Exist

In [ ]:
# Shared perf table — both vLLM_Batch and vLLM_Real_Time write here
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PERF_TABLE} (
  option STRING, tc_id STRING, cold_start_s DOUBLE,
  model_load_s DOUBLE, latency_s DOUBLE, pages LONG, ts TIMESTAMP
) USING DELTA
""")
print(f"Perf table ready: {PERF_TABLE}")

# Parsed results table — stores extracted markdown per test case
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} (
  tc_id STRING, pages LONG, markdown STRING, char_count LONG, ts TIMESTAMP
) USING DELTA
""")
print(f"Results table ready: {RESULTS_TABLE}")

### Wait for vLLM Readiness

Polls the `/v1/models` endpoint until vLLM returns HTTP 200. After readiness, sends a **text-only warmup** request to trigger CUDA kernel JIT compilation before the actual image tests.

**Important notes:**
- vLLM `/models` can return 200 before the vision pipeline is fully loaded — the warmup mitigates this
- HTTP 401 from the driver proxy means the cluster is **terminated** (not an auth error)
- HTTP 502/503 means vLLM is still loading — keep waiting

In [ ]:
# --- Wait for vLLM readiness ---
print(f"Waiting for vLLM at {BASE_URL}/models ...")
deadline = time.time() + READY_TIMEOUT
while time.time() < deadline:
    try:
        r = requests.get(f"{BASE_URL}/models", headers=HEADERS, timeout=10)
        if r.status_code == 200:
            print(f"vLLM ready (API responding): {r.json()['data'][0]['id']}")
            break
        # HTTP 401 = cluster dead (driver proxy returns 401 for terminated clusters)
        if r.status_code == 401:
            raise RuntimeError("HTTP 401 from driver proxy — cluster is terminated or not yet started")
        print(f"  HTTP {r.status_code} — waiting ...")
    except RuntimeError:
        raise
    except Exception as e:
        print(f"  {type(e).__name__} — waiting ...")
    time.sleep(15)
else:
    raise TimeoutError(f"vLLM not ready within {READY_TIMEOUT}s")

# --- Warmup: text-only request to trigger CUDA kernel JIT ---
# The first inference after model load compiles CUDA kernels, which can take
# minutes. A cheap text-only request warms this up before the real tests.
print("Sending warmup inference (text-only) ...")
_warmup_payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Hello"}],
    "max_tokens": 16,
    "temperature": 0.0,
}
try:
    _wr = requests.post(f"{BASE_URL}/chat/completions", json=_warmup_payload,
                        headers=HEADERS, timeout=TEST_TIMEOUT)
    _wr.raise_for_status()
    print(f"Warmup done: {_wr.status_code}")
except requests.HTTPError as _we:
    # Warmup failure is non-fatal — the real tests may still succeed
    _wb = _we.response.text[:300] if _we.response is not None else ""
    print(f"Warmup HTTP error (non-fatal): {_we.response.status_code}: {_wb}")
except Exception as _we:
    print(f"Warmup failed (non-fatal): {_we}")

### Helper Functions

- **`rasterize_pdf`**: Converts PDF bytes to a list of PNG images (one per page) at 150 DPI
- **`query_page`**: Sends a single page image to vLLM via the driver proxy and returns the extracted markdown

In [ ]:
def rasterize_pdf(pdf_bytes: bytes, dpi: int = 150) -> list:
    """Convert PDF bytes to a list of PNG byte arrays, one per page."""
    doc   = fitz.open(stream=pdf_bytes, filetype="pdf")
    scale = dpi / 72.0                          # PDF points → pixels
    mat   = fitz.Matrix(scale, scale)
    pages = [page.get_pixmap(matrix=mat).tobytes("png") for page in doc]
    doc.close()
    return pages

def query_page(img_bytes: bytes) -> str:
    """Send a page image to vLLM and return extracted markdown."""
    img_b64 = base64.b64encode(img_bytes).decode()
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
            {"type": "text", "text": "Extract all text, tables, and structure from this page. Output as clean markdown."},
        ]}],
        "max_tokens": 2048,
        "temperature": 0.0,
    }
    r = requests.post(f"{BASE_URL}/chat/completions", json=payload, headers=HEADERS, timeout=TEST_TIMEOUT)
    if r.status_code != 200:
        print(f"    vLLM error {r.status_code}: {r.text[:500]}")
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

### Run Tests and Record Results

For each test case (TC1–TC4):
1. Read the PDF from the Volume and rasterise to PNGs
2. Send each page to vLLM via the driver proxy
3. Measure wall-clock latency (model is already hot — no cold start)
4. A test **passes** if it returns more than 5 characters of markdown

`cold_start_s` is always 0.0 for vLLM_Real_Time because the model is pre-loaded by the continuous job.

Parsed markdown is saved to the results table. Performance metrics and a pass/fail summary are written to the perf table.

In [ ]:
tc_files = {"TC1": "tc1.pdf", "TC2": "tc2.pdf", "TC3": "tc3.pdf", "TC4": "tc4.pdf"}
results  = {}    # tc_id → "PASS" or "FAIL: reason"
records  = []    # perf rows to write
parsed   = []    # parsed markdown rows to write

for tc_id, fname in tc_files.items():
    pdf_path = os.path.join(TC_PATH, fname)
    doc      = fitz.open(pdf_path); pages = len(doc); doc.close()
    print(f"\n{tc_id} ({pages} pages) ...")
    try:
        with open(pdf_path, "rb") as f:
            pdf_bytes = f.read()
        t0         = time.time()  # Start per-document timer
        page_imgs  = rasterize_pdf(pdf_bytes)
        page_texts = [query_page(img) for img in page_imgs]
        markdown   = "\n\n---\n\n".join(page_texts) if len(page_texts) > 1 else page_texts[0]
        latency    = time.time() - t0  # Per-document wall clock
        ok         = len(markdown) > 5
        results[tc_id] = "PASS" if ok else f"FAIL: {len(markdown)} chars"
        print(f"  {results[tc_id]} — {latency:.1f}s, {len(markdown)} chars")
        records.append({
            "option": "vLLM_Real_Time", "tc_id": tc_id,
            "cold_start_s": 0.0,   # Always 0 — model is pre-loaded
            "model_load_s": 0.0,
            "latency_s": latency,
            "pages": pages, "ts": datetime.now(timezone.utc),
        })
        parsed.append({
            "tc_id": tc_id, "pages": pages,
            "markdown": markdown, "char_count": len(markdown),
            "ts": datetime.now(timezone.utc),
        })
    except requests.HTTPError as e:
        # Capture HTTP response body for debugging (500/502 from vLLM)
        body = e.response.text[:300] if e.response is not None else ""
        results[tc_id] = f"FAIL: {e.response.status_code}: {body}"
        print(f"  {results[tc_id]}")
    except Exception as e:
        import traceback
        results[tc_id] = f"FAIL: {type(e).__name__}: {e}"
        print(f"  {results[tc_id]}")
        traceback.print_exc()

# Write parsed results to the results table
if parsed:
    spark.createDataFrame(parsed).write.mode("append").saveAsTable(RESULTS_TABLE)
    print(f"\nWrote {len(parsed)} parsed result(s) to {RESULTS_TABLE}")

# Write perf records to shared results table
if records:
    spark.createDataFrame(records).write.mode("append").saveAsTable(PERF_TABLE)
    print(f"Wrote {len(records)} perf record(s) to {PERF_TABLE}")

# Print summary banner
print("\n" + "=" * 60)
print("TEST SUMMARY — vLLM_Real_Time")
print("=" * 60)
for tc, res in results.items():
    print(f"  [{'PASS' if res == 'PASS' else 'FAIL'}] {tc}: {res}")
passed = sum(1 for v in results.values() if v == "PASS")
print(f"\n{passed}/{len(results)} tests passed")

# Surface results via Jobs API (serverless stdout is not captured)
import json as _json
dbutils.notebook.exit(_json.dumps({"passed": passed, "total": len(results), "results": results}))